In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, max ,lit , row_number
from pyspark.sql.window import Window

print("Imports successful")

Imports successful


In [0]:
# Step 1 - Read watermark function
def read_watermark(pipeline_name):
    
    watermark_df = spark.sql("""
        SELECT last_watermark
        FROM pipeline_control.watermarks
        WHERE pipeline_name = '{}'
    """.format(pipeline_name))
    
    last_watermark = watermark_df.collect()[0][0]
    
    print(f"Pipeline: {pipeline_name}")
    print(f"Last watermark: {last_watermark}")
    
    return last_watermark

In [0]:
def read_bronze_incrementally(source_table, watermark_col, last_watermark):

    bronze_df = spark.sql(f"""
        SELECT *
        FROM {source_table}
        WHERE {watermark_col} > '{last_watermark}'
        """)
    print(f"Found {bronze_df.count()} new records in {source_table}")
    
    return bronze_df


In [0]:
# Step 3 - Deduplicate records
def deduplicate(df, merge_key, watermark_col):
    
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, col
    
    window = Window \
        .partitionBy(merge_key) \
        .orderBy(col(watermark_col).desc())
    
    deduped_df = df \
        .withColumn("rn", row_number().over(window)) \
        .filter("rn = 1") \
        .drop("rn")
    
    print(f"Records before dedup: {df.count()}")
    print(f"Records after dedup:  {deduped_df.count()}")
    
    return deduped_df

In [0]:
# Step 4 - MERGE into Silver
def merge_into_silver(deduped_df, target_table, merge_key, watermark_col):
    
    silver_table = DeltaTable.forName(spark, target_table)
    
    # Build dynamic update set from source columns
    update_set = {
        c: f"source.{c}" 
        for c in deduped_df.columns 
        if c != merge_key and c != "_source"
    }
    
    # Build dynamic insert values from source columns
    insert_values = {
        c: f"source.{c}" 
        for c in deduped_df.columns 
        if c != "_source"
    }
    
    # Add is_active to both
    update_set["is_active"]    = "true"
    insert_values["is_active"] = "true"
    
    silver_table.alias("target").merge(
        deduped_df.alias("source"),
        f"target.{merge_key} = source.{merge_key}"
    ) \
    .whenMatchedUpdate(
        condition=f"source.{watermark_col} > target.{watermark_col}",
        set=update_set
    ) \
    .whenNotMatchedInsert(
        values=insert_values
    ) \
    .execute()
    
    print(f"MERGE complete into {target_table}")

In [0]:
# Step 5 - Update watermark
def update_watermark(pipeline_name, deduped_df, watermark_col):
    
    # Find highest timestamp in processed batch
    new_watermark = deduped_df \
        .agg(max(watermark_col)) \
        .collect()[0][0]
    
    # Update watermark table
    spark.sql(f"""
        UPDATE pipeline_control.watermarks
        SET last_watermark = '{new_watermark}',
            updated_at = current_timestamp()
        WHERE pipeline_name = '{pipeline_name}'
    """)
    
    print(f"Watermark updated to: {new_watermark}")

In [0]:
# Master function - runs all 5 steps
def run_bronze_to_silver(pipeline_name, source_table,
                          target_table, merge_key, watermark_col):
    
    print(f"Starting pipeline: {pipeline_name}")
    print("=" * 50)
    
    # Step 1 - Read watermark
    last_watermark = read_watermark(pipeline_name)
    
    # Step 2 - Read Bronze incrementally
    bronze_df = read_bronze_incrementally(
        source_table,
        watermark_col,
        last_watermark
    )
    
    # Step 3 - Check if data exists
    if bronze_df.count() == 0:
        print(f"No new records found — skipping")
        print("=" * 50)
        return
    
    # Step 4 - Remove PII for members table
    if "members" in source_table:
        bronze_df = bronze_df.drop(
            "first_name", 
            "last_name", 
            "address",
            "patient_id"
        )
        print("PII removed from members data")
    
    # Step 5 - Deduplicate
    deduped_df = deduplicate(bronze_df, merge_key, watermark_col)
    
    # Step 6 - MERGE into Silver
    merge_into_silver(deduped_df, target_table, merge_key, watermark_col)
    
    # Step 7 - Update watermark
    update_watermark(pipeline_name, deduped_df, watermark_col)
    
    print("=" * 50)
    print(f"Pipeline complete: {pipeline_name}")

In [0]:
# Add sample data for providers
providers_data = [
    ("PRV01", "Dr. John Smith",    "Cardiology",   "New York, NY",    "2024-01-15 10:00:00"),
    ("PRV02", "Dr. Sarah Johnson", "Pediatrics",   "Los Angeles, CA", "2024-01-15 11:00:00"),
    ("PRV03", "Dr. Mike Williams", "Orthopedics",  "Chicago, IL",     "2024-01-15 12:00:00"),
]

providers_df = spark.createDataFrame(
    providers_data,
    ["provider_id", "provider_name", "specialty", "location", "updated_at"]
) \
.withColumn("updated_at",   col("updated_at").cast("timestamp")) \
.withColumn("_ingested_at", current_timestamp()) \
.withColumn("_source",      lit("SFTP"))

providers_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.providers")

print(f"Landed {providers_df.count()} providers in Bronze")

# Add sample data for payments
payments_data = [
    ("PAY001", "CLM001", 1500.00, "2024-01-16", "PROCESSED", "2024-01-16 10:00:00"),
    ("PAY002", "CLM002",  800.00, "2024-01-16", "PENDING",   "2024-01-16 11:00:00"),
    ("PAY003", "CLM004", 3100.00, "2024-01-16", "PROCESSED", "2024-01-16 12:00:00"),
]

payments_df = spark.createDataFrame(
    payments_data,
    ["payment_id", "claim_id", "payment_amount", "payment_date", "payment_status", "created_at"]
) \
.withColumn("created_at",   col("created_at").cast("timestamp")) \
.withColumn("payment_date", col("payment_date").cast("date")) \
.withColumn("_ingested_at", current_timestamp()) \
.withColumn("_source",      lit("SFTP"))

payments_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.payments")

print(f"Landed {payments_df.count()} payments in Bronze")

# Add sample data for eligibility
eligibility_data = [
    ("ELG001", "PAT101", "PLAN001", "2023-01-01", "2024-12-31", "MEDICAL",  "2024-01-15 10:00:00"),
    ("ELG002", "PAT102", "PLAN002", "2023-06-01", "2024-12-31", "DENTAL",   "2024-01-15 11:00:00"),
    ("ELG003", "PAT103", "PLAN001", "2022-01-01", "2024-12-31", "MEDICAL",  "2024-01-15 12:00:00"),
    ("ELG004", "PAT104", "PLAN003", "2023-03-01", "2024-12-31", "VISION",   "2024-01-15 13:00:00"),
    ("ELG005", "PAT105", "PLAN002", "2023-09-01", "2024-12-31", "MEDICAL",  "2024-01-15 14:00:00"),
]

eligibility_df = spark.createDataFrame(
    eligibility_data,
    ["eligibility_id", "member_id", "plan_id", "effective_date", 
     "termination_date", "coverage_type", "updated_at"]
) \
.withColumn("updated_at",        col("updated_at").cast("timestamp")) \
.withColumn("effective_date",    col("effective_date").cast("date")) \
.withColumn("termination_date",  col("termination_date").cast("date")) \
.withColumn("_ingested_at",      current_timestamp()) \
.withColumn("_source",           lit("SFTP"))

eligibility_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.eligibility")

print(f"Landed {eligibility_df.count()} eligibility records in Bronze")

Landed 3 providers in Bronze
Landed 3 payments in Bronze
Landed 5 eligibility records in Bronze


In [0]:
# Run metadata-driven pipeline for all 5 tables
config_df = spark.sql("""
    SELECT * 
    FROM pipeline_control.ingestion_config
    WHERE is_active = true
""")

for row in config_df.collect():
    source_name   = row["source_name"]
    source_table  = row["source_path"]
    target_table  = row["target_table"]
    merge_key     = row["merge_key"]
    watermark_col = row["watermark_col"]
    pipeline_name = f"{source_name}_silver"
    
    run_bronze_to_silver(
        pipeline_name = pipeline_name,
        source_table  = source_table,
        target_table  = target_table,
        merge_key     = merge_key,
        watermark_col = watermark_col
    )

Starting pipeline: claims_silver
Pipeline: claims_silver
Last watermark: 2024-01-16 12:00:00
Found 0 new records in bronze.claims
No new records found — skipping
Starting pipeline: members_silver
Pipeline: members_silver
Last watermark: 2024-01-15 14:00:00
Found 0 new records in bronze.members
No new records found — skipping
Starting pipeline: providers_silver
Pipeline: providers_silver
Last watermark: 2024-01-15 12:00:00
Found 0 new records in bronze.providers
No new records found — skipping
Starting pipeline: payments_silver
Pipeline: payments_silver
Last watermark: 2020-01-01 00:00:00
Found 3 new records in bronze.payments
Records before dedup: 3
Records after dedup:  3
MERGE complete into silver.payments
Watermark updated to: 2024-01-16 12:00:00
Pipeline complete: payments_silver
Starting pipeline: eligibility_silver
Pipeline: eligibility_silver
Last watermark: 2020-01-01 00:00:00
Found 5 new records in bronze.eligibility
Records before dedup: 5
Records after dedup:  5
MERGE comple

In [0]:
# Check all Silver table counts
print("=== SILVER TABLE COUNTS ===")
spark.sql("SELECT COUNT(*) as claims_count FROM silver.claims").show()
spark.sql("SELECT COUNT(*) as members_count FROM silver.members").show()
spark.sql("SELECT COUNT(*) as providers_count FROM silver.providers").show()
spark.sql("SELECT COUNT(*) as payments_count FROM silver.payments").show()
spark.sql("SELECT COUNT(*) as eligibility_count FROM silver.eligibility").show()

=== SILVER TABLE COUNTS ===
+------------+
|claims_count|
+------------+
|           8|
+------------+

+-------------+
|members_count|
+-------------+
|            5|
+-------------+

+---------------+
|providers_count|
+---------------+
|              3|
+---------------+

+--------------+
|payments_count|
+--------------+
|             3|
+--------------+

+-----------------+
|eligibility_count|
+-----------------+
|                5|
+-----------------+



In [0]:
# Verify PII removed from members
print("=== SILVER MEMBERS — NO PII ===")
spark.sql("SELECT * FROM silver.members").show()

=== SILVER MEMBERS — NO PII ===
+---------+-------------+-------------------+---------+--------------------+
|member_id|date_of_birth|         updated_at|is_active|        _ingested_at|
+---------+-------------+-------------------+---------+--------------------+
|   MEM001|   1980-05-15|2024-01-15 10:00:00|     true|2026-09-11 01:11:...|
|   MEM002|   1990-03-20|2024-01-15 11:00:00|     true|2026-09-11 01:11:...|
|   MEM003|   1975-08-10|2024-01-15 12:00:00|     true|2026-09-11 01:11:...|
|   MEM004|   1985-12-25|2024-01-15 13:00:00|     true|2026-09-11 01:11:...|
|   MEM005|   1970-06-30|2024-01-15 14:00:00|     true|2026-09-11 01:11:...|
+---------+-------------+-------------------+---------+--------------------+

